# GDP Preprocessing — DRC (COD) 2024

Two extraction strategies from the Kummu v4 dataset:
- **Windowed** (): memory-light, recommended for large global rasters
- **General extractor** (): auto-downloads boundary, clips, saves

In [1]:
import os, sys
ROOT_DIR = os.path.abspath(os.curdir)
sys.path.insert(0, ROOT_DIR)

from preprocess_kummu_gdp import download_file, preprocess_kummu_gdp_tif_windowed, KUMMU_V4_FILES
from extract_gdp import extract_country_gdp
import geopandas as gpd

gdp_path = os.path.join(ROOT_DIR, "GDP_data")
admin_path = os.path.join(ROOT_DIR, "admin")
os.makedirs(gdp_path, exist_ok=True)

In [2]:
# Download Kummu v4 30 arc-sec total GDP raster (1.4 GB — skips if already present)
tif_path = download_file(
    url=KUMMU_V4_FILES["rast_gdpTot_1990_2024_30arcsec.tif"],
    out_dir=gdp_path,
)
print("Source raster:", tif_path)

Already exists, skipping download: C:\Users\Bachirou\Documents\CREDIT_Model_DRC\Credit_DRC\GDP_data\rast_gdpTot_1990_2024_30arcsec.tif
Source raster: C:\Users\Bachirou\Documents\CREDIT_Model_DRC\Credit_DRC\GDP_data\rast_gdpTot_1990_2024_30arcsec.tif


## Strategy 1 — Windowed extraction (requires local boundary shapefile)

In [3]:
# Load DRC ADM1 boundary (downloads from geoBoundaries if not present)
from extract_gdp import _get_country_boundary

_shp = os.path.join(admin_path, "DRC_admin1.shp")
if not os.path.exists(_shp):
    os.makedirs(admin_path, exist_ok=True)
    cod_gdf = _get_country_boundary("COD", adm_level="ADM1")
    cod_gdf.to_file(_shp)
    print(f"Downloaded and saved {len(cod_gdf)} provinces to {_shp}")
else:
    cod_gdf = gpd.read_file(_shp)
    print(f"Loaded {len(cod_gdf)} provinces from {_shp}")

da_windowed = preprocess_kummu_gdp_tif_windowed(
    tif_path=tif_path,
    year=2024,
    boundary_gdf=cod_gdf,
    out_path=os.path.join(gdp_path, "GDP_PPP_2024_COD_windowed.tif"),
)
print("Total DRC GDP 2024 (windowed):", float(da_windowed.sum()))

Loaded 26 provinces from C:\Users\Bachirou\Documents\CREDIT_Model_DRC\Credit_DRC\admin\DRC_admin1.shp
Window bounds (padded ±0.05°): (12.151, -13.509, 31.3274, 5.442)
No usable embedded band-year tags; assuming order: (1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024)
  -> VERIFY this matches github.com/mattikummu/griddedGDPpc before trusting results.
Reading window: Window(col_off=23059, row_off=10038, width=2301, height=2274) (band 8 -> year 2024)
Window shape: (2274, 2301) | valid pixels: 5123054 / 5232474
Post-clip shape: (2262, 2289) | valid: 2712062 / 5177718
Saved: C:\Users\Bachirou\Documents\CREDIT_Model_DRC\Credit_DRC\GDP_data\GDP_PPP_2024_COD_windowed.tif
Total DRC GDP 2024 (windowed): 189951377408.0


## Strategy 2 — General extractor (auto-downloads boundary from geoBoundaries API)

In [8]:
da = extract_country_gdp(
    country="COD",
    year=2024,
    gdp_path=gdp_path,
    out_path=os.path.join(gdp_path, "GDP_PPP_2024_COD.tif"),
)
print("Total DRC GDP 2024 (general extractor):", float(da.sum()))

COD (COD) 2024: 501834 valid pixels, total GDP=189,951,377,408
Saved: C:\Users\Bachirou\Documents\CREDIT_Model_DRC\Credit_DRC\GDP_data\GDP_PPP_2024_COD.tif
Total DRC GDP 2024 (general extractor): 189951377408.0
